# TREAT-MMTB 2026 — Task 1 Baseline Detector

**This is the floor, not OrdFused-CXR.** Single in-domain encoder + binary head. Every later component (ordinal head, gated metadata fusion, multi-encoder fusion, ensemble) is measured as a Δ against the number this notebook produces.

What this notebook gives you:
1. **SimpleITK-first** image+mask loading with a geometry-verification QC step (the organizers explicitly warn that pydicom+nibabel loading causes flips).
2. CXR-appropriate preprocessing: percentile clip → CLAHE → resize → normalize → 3-channel.
3. **Patient-grouped, grade-stratified** split (no patient leakage; preserves none/small/medium/large ratios).
4. Ordinal-aware label encoding kept in the loader so the upgrade to the CORN head is one line.
5. The **exact challenge metric** `0.7·Accuracy + 0.3·Dice` implemented, with τ-sweeping on validation.

> Replace the three paths in the CONFIG cell with the real dataset when it drops (May 22). Everything else runs unchanged.

Known internal-set distribution (from the clinical CSV): **none 247 / small 79 / medium 61 / large 57 → 444 total, 44.4% positive.**

## 0. Environment

In [1]:
# Core
!pip install -q SimpleITK opencv-python-headless pandas numpy scikit-learn matplotlib timm torch torchvision
# EVA-X / RAD-DINO are loaded from timm or HF; see the encoder cell for options.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
monai 1.4.0 requires numpy<2.0,>=1.24, but you have numpy 2.2.6 which is incompatible.


In [2]:
import os, glob, random, math
import numpy as np
import pandas as pd
import SimpleITK as sitk
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


## 1. CONFIG — edit these paths for the real dataset

In [ ]:
class CFG:
    # --- paths (EDIT on data release) ---
    csv_path   = 'Data/test.csv'          # clinical CSV with _id, cavity, age_of_onset, gender, series_modality_cd, ...
    image_dir  = '/Data/train/CXR'             # CXR files, named by _id (e.g. 2.dcm / 2.png / 2.nii.gz)
    mask_dir   = '/Data/train/CXR_label'              # segmentation masks, named by _id; empty/all-zero when no cavity
    image_ext  = '.dcm'                 # '.dcm' | '.png' | '.nii.gz'  (loader auto-detects too)
    mask_ext   = '.nii.gz'

    # --- preprocessing ---
    img_size   = 224                    # ViT-B native (EVA-X / RAD-DINO). 384/512 for higher-res encoders.
    clip_lo, clip_hi = 1.0, 99.0        # percentile clip to remove detector-saturation outliers
    use_clahe  = True
    clahe_clip = 2.0
    clahe_grid = 8
    # ImageNet stats work for most timm/HF pretrained ViTs; swap to CXR stats if the encoder card specifies.
    norm_mean  = (0.485, 0.456, 0.406)
    norm_std   = (0.229, 0.224, 0.225)

    # --- split / train ---
    n_folds    = 5
    fold       = 0                      # which fold is validation this run
    batch_size = 16
    epochs     = 25
    lr         = 3e-4
    weight_decay = 1e-4
    encoder    = 'convnext_tiny'        # BASELINE encoder; see encoder cell to switch to EVA-X / RAD-DINO
    freeze_encoder = False              # baseline fine-tunes; OrdFused freezes + LoRA

cfg = CFG()
GRADE2ORD = {'none': 0, 'small': 1, 'medium': 2, 'large': 3}   # ordinal rank
ORD2GRADE = {v: k for k, v in GRADE2ORD.items()}

## 2. SimpleITK loading + geometry verification

**This is the single most important defensive step.** Loading image and mask with different libraries flips them silently and destroys Dice without any error. We load **both** with SimpleITK and assert their geometry matches before anything else touches them.

In [28]:
def sitk_read(path):
    """Load any of dcm/png/nii.gz to a 2D float32 numpy array via SimpleITK."""
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)          # (z,y,x) or (y,x)
    if arr.ndim == 3:
        arr = arr[0] if arr.shape[0] < arr.shape[-1] else arr[..., 0]
    return arr.astype(np.float32), img

def verify_geometry(img_sitk, mask_sitk, atol=1e-3):
    """Assert image and mask share spacing/origin/direction/size. Returns list of mismatches."""
    problems = []
    if img_sitk.GetSize() != mask_sitk.GetSize():
        problems.append(f"size {img_sitk.GetSize()} vs {mask_sitk.GetSize()}")
    if not np.allclose(img_sitk.GetSpacing(), mask_sitk.GetSpacing(), atol=atol):
        problems.append(f"spacing {img_sitk.GetSpacing()} vs {mask_sitk.GetSpacing()}")
    if not np.allclose(img_sitk.GetOrigin(), mask_sitk.GetOrigin(), atol=atol):
        problems.append(f"origin {img_sitk.GetOrigin()} vs {mask_sitk.GetOrigin()}")
    if not np.allclose(img_sitk.GetDirection(), mask_sitk.GetDirection(), atol=atol):
        problems.append(f"direction mismatch")
    return problems

def resample_mask_to_image(img_sitk, mask_sitk):
    """If geometry differs, resample mask onto the image grid (nearest-neighbor)."""
    rs = sitk.ResampleImageFilter()
    rs.SetReferenceImage(img_sitk)
    rs.SetInterpolator(sitk.sitkNearestNeighbor)
    rs.SetDefaultPixelValue(0)
    return rs.Execute(mask_sitk)

## 3. Visual QC — overlay mask on image for a random sample (run on day 1)

In [29]:
def find_file(d, _id, ext):
    for cand in [os.path.join(d, f'{_id}{ext}'),
                 os.path.join(d, f'{_id}.png'),
                 os.path.join(d, f'{_id}.dcm'),
                 os.path.join(d, f'{_id}.nii.gz')]:
        if os.path.exists(cand):
            return cand
    hits = glob.glob(os.path.join(d, f'{_id}.*'))
    return hits[0] if hits else None

def qc_overlay(ids, n=8):
    ids = list(ids)[:n]
    fig, ax = plt.subplots(2, (n+1)//2, figsize=(3*((n+1)//2), 6))
    ax = ax.ravel()
    for i, _id in enumerate(ids):
        ip = find_file(cfg.image_dir, _id, cfg.image_ext)
        mp = find_file(cfg.mask_dir, _id, cfg.mask_ext)
        arr, img_s = sitk_read(ip)
        if mp:
            m_arr, mask_s = sitk_read(mp)
            probs = verify_geometry(img_s, mask_s)
            if probs:
                mask_s = resample_mask_to_image(img_s, mask_s)
                m_arr = sitk.GetArrayFromImage(mask_s).astype(np.float32)
                if m_arr.ndim == 3: m_arr = m_arr[0]
                title = f'{_id} [resampled]'
            else:
                title = f'{_id} [ok]'
        else:
            m_arr = np.zeros_like(arr); title = f'{_id} [no mask]'
        a = np.clip(arr, np.percentile(arr,1), np.percentile(arr,99))
        a = (a - a.min())/(a.ptp()+1e-6)
        ax[i].imshow(a, cmap='gray')
        ax[i].imshow(np.ma.masked_where(m_arr<0.5, m_arr), cmap='autumn', alpha=0.45)
        ax[i].set_title(title, fontsize=9); ax[i].axis('off')
    plt.tight_layout(); plt.show()

df = pd.read_csv(cfg.csv_path)
pos_ids = df[df.cavity != 'none']._id.tolist()
qc_overlay(pos_ids, n=8)   # <-- LOOK at this. If masks are off the lesion, fix loading before training.

AttributeError: 'DataFrame' object has no attribute '_id'

## 4. Preprocessing: clip → CLAHE → resize → normalize → 3-channel

In [6]:
def preprocess_image(arr):
    # 1) percentile clip (robust to detector saturation / burned-in markers)
    lo, hi = np.percentile(arr, cfg.clip_lo), np.percentile(arr, cfg.clip_hi)
    arr = np.clip(arr, lo, hi)
    # 2) to 0..255 uint8 for CLAHE
    arr = (arr - arr.min()) / (arr.ptp() + 1e-6)
    u8 = (arr * 255).astype(np.uint8)
    # 3) CLAHE (local contrast; well-supported for TB-CXR)
    if cfg.use_clahe:
        clahe = cv2.createCLAHE(clipLimit=cfg.clahe_clip,
                                tileGridSize=(cfg.clahe_grid, cfg.clahe_grid))
        u8 = clahe.apply(u8)
    # 4) resize
    u8 = cv2.resize(u8, (cfg.img_size, cfg.img_size), interpolation=cv2.INTER_AREA)
    # 5) normalize + 3-channel replicate for ImageNet/CXR-pretrained ViTs
    x = u8.astype(np.float32) / 255.0
    x = np.stack([x, x, x], 0)                     # (3,H,W)
    mean = np.array(cfg.norm_mean).reshape(3,1,1)
    std  = np.array(cfg.norm_std).reshape(3,1,1)
    x = (x - mean) / std
    return x.astype(np.float32)

## 5. Dataset — returns image, binary label, ordinal label, and mask presence

In [7]:
class CavityDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train
    def __len__(self):
        return len(self.df)
    def _augment(self, x):
        if random.random() < 0.5:                  # h-flip
            x = x[:, :, ::-1].copy()
        return x
    def __getitem__(self, i):
        row = self.df.iloc[i]
        ip = find_file(cfg.image_dir, row._id, cfg.image_ext)
        arr, _ = sitk_read(ip)
        x = preprocess_image(arr)
        if self.train:
            x = self._augment(x)
        ordl = GRADE2ORD[row.cavity]               # 0..3, ordinal
        binl = 1 if ordl > 0 else 0                 # scored label
        return (torch.from_numpy(x),
                torch.tensor(binl, dtype=torch.float32),
                torch.tensor(ordl, dtype=torch.long))

## 6. Patient-grouped, grade-stratified split

If the CSV has no patient/group id, we fall back to `_id` as the group (one image per case). Swap in the real patient column when known — it prevents the same patient landing in both train and val, which would inflate your score.

In [26]:
def make_folds(df, group_col=None):
    df = df.copy()
    df['ord'] = df.cavity.map(GRADE2ORD)
    groups = df[group_col] if group_col else df['_id']
    sgkf = StratifiedGroupKFold(n_splits=cfg.n_folds, shuffle=True, random_state=SEED)
    df['fold'] = -1
    for f, (_, val_idx) in enumerate(sgkf.split(df, df['ord'], groups)):
        df.loc[df.index[val_idx], 'fold'] = f
    return df
df = pd.read_csv(cfg.csv_path)
df = make_folds(df, group_col=None)   # set group_col='patient_id' if it exists
print(df.groupby(['fold','cavity']).size().unstack(fill_value=0))

KeyError: '_id'

## 7. Baseline model — single encoder + binary head

Baseline uses ConvNeXt-Tiny for a fast, dependency-light floor. To switch to the in-domain encoders from the method doc, see the commented options — the rest of the notebook is unchanged.

In [9]:
import timm

class BaselineDetector(nn.Module):
    def __init__(self, name='convnext_tiny', pretrained=True, freeze=False):
        super().__init__()
        self.encoder = timm.create_model(name, pretrained=pretrained,
                                         num_classes=0, global_pool='avg')
        feat = self.encoder.num_features
        if freeze:
            for p in self.encoder.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(nn.LayerNorm(feat), nn.Dropout(0.2), nn.Linear(feat, 1))
    def forward(self, x):
        z = self.encoder(x)
        return self.head(z).squeeze(1)             # logit for P(cavity)

# --- In-domain encoder options (method doc) ---
# EVA-X:   timm.create_model('eva02_base_patch14_224', ...) then load EVA-X SSL weights
#          (github.com/hustvl/EVA-X — download checkpoint, load_state_dict)
# RAD-DINO: from transformers import AutoModel; AutoModel.from_pretrained('microsoft/rad-dino')
#          (wrap so forward() returns the pooled CLS embedding)
# For OrdFused you FREEZE these and add LoRA; the baseline just needs one working encoder.

model = BaselineDetector(cfg.encoder, pretrained=True, freeze=cfg.freeze_encoder).to(DEVICE)
print(sum(p.numel() for p in model.parameters())/1e6, 'M params')

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

27.822433 M params


## 8. The challenge metric — implemented exactly

Score = `0.7 · Detection Accuracy + 0.3 · Dice`. For the **detection-only baseline** we compute the Dice term as the *presence-consistent* Dice (a correctly-detected-absent case scores Dice=1 by convention of empty==empty; a false positive scores 0). When the segmentor is wired in, replace `dice_from_masks` with true pixel Dice — the accuracy term and τ-sweep are unchanged.

In [10]:
def detection_accuracy(y_true, y_pred):
    return float((y_true == y_pred).mean())

def presence_dice(y_true, y_pred):
    """Placeholder Dice term for detection-only baseline.
    empty-vs-empty (TN) -> 1 ; correct positive -> use real mask Dice later ;
    false pos / false neg -> 0. This gives a metric-shaped proxy so tau tuning
    already optimizes the right objective before the segmentor exists."""
    d = np.zeros(len(y_true))
    d[(y_true==0)&(y_pred==0)] = 1.0
    d[(y_true==1)&(y_pred==1)] = 0.60   # assume field-level ~0.60 pixel Dice on true positives
    return float(d.mean())

def challenge_score(y_true, prob, tau):
    y_pred = (prob >= tau).astype(int)
    acc = detection_accuracy(y_true, y_pred)
    dice = presence_dice(y_true, y_pred)
    return 0.7*acc + 0.3*dice, acc, dice

def sweep_tau(y_true, prob, grid=None):
    grid = grid if grid is not None else np.linspace(0.05, 0.95, 181)
    best = max(((challenge_score(y_true, prob, t)[0], t) for t in grid))
    return best[1], best[0]                        # tau*, score*

## 9. Train / validate one fold

In [25]:
def run_fold(df):
    tr = df[df.fold != cfg.fold]; va = df[df.fold == cfg.fold]
    # class-balanced positive weight for BCE (near-balanced set, so mild)
    pos = (tr.cavity != 'none').sum(); neg = (tr.cavity == 'none').sum()
    pos_weight = torch.tensor([neg/max(pos,1)], device=DEVICE)
    dl_tr = DataLoader(CavityDataset(tr, True),  batch_size=cfg.batch_size, shuffle=True,  num_workers=2, drop_last=True)
    dl_va = DataLoader(CavityDataset(va, False), batch_size=cfg.batch_size, shuffle=False, num_workers=2)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    lossfn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best = {'score': -1}
    for ep in range(cfg.epochs):
        model.train()
        for x, yb, yo in dl_tr:
            x, yb = x.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit = model(x)
            loss = lossfn(logit, yb)
            loss.backward(); opt.step()
        sched.step()
        # validate
        model.eval(); probs=[]; ys=[]
        with torch.no_grad():
            for x, yb, yo in dl_va:
                p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
                probs.append(p); ys.append(yb.numpy())
        probs = np.concatenate(probs); ys = np.concatenate(ys).astype(int)
        tau, score = sweep_tau(ys, probs)
        s, acc, dice = challenge_score(ys, probs, tau)
        if score > best['score']:
            best = {'score':score, 'tau':tau, 'acc':acc, 'dice':dice, 'ep':ep}
            torch.save(model.state_dict(), 'baseline_best.pt')
        print(f'ep{ep:02d}  score={score:.4f}  acc={acc:.4f}  dice={dice:.4f}  tau*={tau:.3f}')
    print('BEST:', best)
    return best

df = pd.read_csv(cfg.csv_path)
df = make_folds(df)
best = run_fold(df)

KeyError: '_id'

## 10. Next steps (the experiment ladder)

This baseline number is row 1 of your ablation table. Then, in order:

1. **Swap encoder** ConvNeXt → EVA-X / RAD-DINO (freeze + LoRA). Measure Δ.
2. **Binary head → CORN ordinal head** (train on `yo`, collapse `P(cavity)=1−P(none)`). Measure Δ. *Expected biggest single jump.*
3. **Add confounder-gated metadata** (age/gender/modality → tabular MLP → gate). Report learned `g`.
4. **Multi-encoder fusion** (late ensemble first as the safe floor; gated feature-fusion as an ablation arm).
5. **Wire the real segmentor** and replace `presence_dice` with true pixel Dice; re-sweep τ end-to-end.

Keep `sweep_tau` and `challenge_score` fixed across all rows so every Δ is measured against the identical objective.

In [13]:
!ls

@eaDir	Code  Data  EDA.ipynb  ordfused_baseline.ipynb
